# 05 Evaluation

This notebook evaluates the poetry graph pipeline from the saved annotation files and processed project outputs.

The evaluation folder only stores gold standard or annotation CSV files. Metric tables are calculated inside this notebook and are not saved as separate CSV files.


## Annotation Files

The notebook uses this annotation file.

- blind_gold_poem_sample.csv contains sampled poems with proxy gold symbols and emotion categories.

This file is kept because it is the evidence behind the evaluation. Summary metrics are calculated live from it.

In [1]:
from pathlib import Path
import re
import pandas as pd

ROOT = Path('..').resolve()
PROCESSED_DIR = ROOT / 'data' / 'processed'
EVALUATION_DIR = ROOT / 'outputs' / 'evaluation'

POEMS_PATH = PROCESSED_DIR / 'poems_clean.csv'
SYMBOLS_PATH = PROCESSED_DIR / 'extracted_symbols.csv'
EMOTIONS_PATH = PROCESSED_DIR / 'extracted_emotions.csv'
RELATIONS_PATH = PROCESSED_DIR / 'symbol_emotion_edges.csv'
GRAPH_NODES_PATH = PROCESSED_DIR / 'graph_nodes.csv'
GRAPH_EDGES_PATH = PROCESSED_DIR / 'graph_edges.csv'

pd.set_option('display.max_colwidth', 140)


## Helper Functions

These functions normalize labels and calculate precision, recall, and F1 from item sets.


In [2]:
def normalize_value(value):
    text = re.sub(r'\s+', ' ', str(value).lower()).strip()
    return re.sub(r'^[^a-z]+|[^a-z]+$', '', text)

def split_label_list(value):
    if pd.isna(value) or not str(value).strip():
        return set()
    return {normalize_value(item) for item in str(value).split(',') if normalize_value(item)}

def set_metrics(gold_sets, predicted_sets):
    rows = []
    total_true_positive = 0
    total_false_positive = 0
    total_false_negative = 0
    for poem_id, gold in gold_sets.items():
        predicted = predicted_sets.get(poem_id, set())
        true_positive = len(gold & predicted)
        false_positive = len(predicted - gold)
        false_negative = len(gold - predicted)
        total_true_positive += true_positive
        total_false_positive += false_positive
        total_false_negative += false_negative
        rows.append({
            'poem_id': poem_id,
            'true_positive': true_positive,
            'false_positive': false_positive,
            'false_negative': false_negative,
            'gold_count': len(gold),
            'predicted_count': len(predicted),
            'matched_items': ', '.join(sorted(gold & predicted)),
            'missed_items': ', '.join(sorted(gold - predicted)),
            'extra_items': ', '.join(sorted(predicted - gold))
        })
    precision = total_true_positive / (total_true_positive + total_false_positive) if total_true_positive + total_false_positive else 0
    recall = total_true_positive / (total_true_positive + total_false_negative) if total_true_positive + total_false_negative else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    summary = pd.DataFrame([{
        'true_positive': total_true_positive,
        'false_positive': total_false_positive,
        'false_negative': total_false_negative,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }])
    return summary, pd.DataFrame(rows)


## Gold Standard Sample

This table is the retained gold standard file for symbol and emotion evaluation. The labels were created from poem text without using the extracted output columns.


In [3]:
gold_sample = pd.read_csv(EVALUATION_DIR / 'blind_gold_poem_sample.csv')
gold_sample[['poem_id', 'title', 'author', 'gold_symbols', 'gold_emotion_categories', 'annotation_method']].head(10)


,poem_id,title,author,gold_symbols,gold_emotion_categories,annotation_method
0,poem_11269,Minor Poet,Bill Sweeney,"mother, night, head, tide, beach, year, plague, brother","hope, joy, mortality",blind_objective_proxy_from_poem_text
1,poem_05452,Wild Poppies,Marion McCready,"throat, bull, night, neck, silk, breadth, pyrenees, opera","despair, envy, freedom, hope, love, melancholy, resilience",blind_objective_proxy_from_poem_text
2,poem_12480,I Heard an Angel,William Blake,"sun, curse, hay, haycock, devil, heath, pourd, grain","freedom, joy, melancholy, peace, tenderness, transcendence",blind_objective_proxy_from_poem_text
3,poem_06414,An Anthology of Rain,Phillis Levin,"drop, collection, limit, air, water, need, devoid, value","anxiety, freedom, gratitude, melancholy, mortality, peace",blind_objective_proxy_from_poem_text
4,poem_11839,Sonnet of the Seven Chinese,Franco Fortini,"image, wall, poet, room, print, photo, worker, lens","doubt, nostalgia",blind_objective_proxy_from_poem_text
5,poem_09632,"from Inscriptions, 16: ""The lamps are burning in the synagogue""",Charles Reznikoff,"morning, eli, answer, lamp, synagogue, house, study, alley","despair, doubt, faith, fear, hope, love, mortality, nostalgia, regret, resilience, transcendence, wonder",blind_objective_proxy_from_poem_text
6,poem_10031,In Memoriam Mae Noblitt,A. R. Ammons,"current, reality, atmosphere, season, space, heel, silk, air","anxiety, fear, freedom, grief, hope, longing, love, nostalgia",blind_objective_proxy_from_poem_text
7,poem_01828,Incident,Natasha Trethewey,"story, window, grass, cross, tree, room, lamp, gown","envy, fear, hope, melancholy, nostalgia, peace, transcendence",blind_objective_proxy_from_poem_text
8,poem_01791,Conduct,Samuel Greenberg,"peninsula, painter, grove, apostle, alm, meek, volcano, sulphur","desire, longing, peace, transcendence",blind_objective_proxy_from_poem_text
9,poem_11907,“Some motionless conﬂict in the sky...”,Donald Revell,"cloud, motionless, skya, universalsimply, saidshe, color, house, rochester","joy, transcendence",blind_objective_proxy_from_poem_text


## Symbol Extraction Evaluation

The system symbols are compared with the gold symbols from the sampled poems. The metric table is calculated here and not saved separately.


In [4]:
symbols_df = pd.read_csv(SYMBOLS_PATH)
gold_symbol_sets = dict(zip(gold_sample['poem_id'], gold_sample['gold_symbols'].map(split_label_list)))
predicted_symbol_sets = symbols_df[symbols_df['poem_id'].isin(gold_symbol_sets)].groupby('poem_id')['symbol'].apply(lambda values: {normalize_value(value) for value in values.dropna() if normalize_value(value)}).to_dict()
symbol_summary, symbol_details = set_metrics(gold_symbol_sets, predicted_symbol_sets)
display(symbol_summary)
symbol_details.head(10)


,true_positive,false_positive,false_negative,precision,recall,f1
0,108,45,132,0.705882,0.45,0.549618


,poem_id,true_positive,false_positive,false_negative,gold_count,predicted_count,matched_items,missed_items,extra_items
0,poem_11269,3,3,5,8,6,"brother, mother, tide","beach, head, night, plague, year","picture, room, track"
1,poem_05452,6,2,2,8,8,"breadth, bull, neck, opera, silk, throat","night, pyrenees","anemone, earth"
2,poem_12480,1,2,7,8,3,heath,"curse, devil, grain, hay, haycock, pourd, sun","mercy, rain"
3,poem_06414,3,1,5,8,4,"air, value, water","collection, devoid, drop, limit, need",request
4,poem_11839,7,0,1,8,7,"image, lens, poet, print, room, wall, worker",photo,
5,poem_09632,3,2,5,8,5,"alley, lamp, synagogue","answer, eli, house, morning, study","cattle, prayer"
6,poem_10031,5,0,3,8,5,"air, atmosphere, season, silk, space","current, heel, reality",
7,poem_01828,6,1,2,8,7,"grass, lamp, room, story, tree, window","cross, gown",angel
8,poem_01791,1,2,7,8,3,painter,"alm, apostle, grove, meek, peninsula, sulphur, volcano","air, ore"
9,poem_11907,3,2,5,8,5,"cloud, color, motionless","house, rochester, saidshe, skya, universalsimply","air, angel"


### Result Interpretation

Symbol extraction achieved 108 true positives, 45 false positives, and 132 false negatives, giving a precision of 0.71, recall of 0.45, and F1 score of 0.55. This means the extractor is reasonably accurate when it identifies a word as a symbol, since about 71% of extracted symbols match the gold labels, but it is also conservative because it misses more than half of the expected symbols. The result shows a clear trade-off from the stricter filtering: the app now produces cleaner and safer public-facing symbols with fewer noisy false positives, but at the cost of lower coverage. Overall, the symbol extraction quality is moderate, with acceptable precision for an interactive poetry app, while recall remains the main weakness.

## Emotion Extraction Evaluation

The system emotion categories are compared with the gold emotion categories from the same sampled poems.


In [5]:
emotions_df = pd.read_csv(EMOTIONS_PATH)
gold_emotion_sets = dict(zip(gold_sample['poem_id'], gold_sample['gold_emotion_categories'].map(split_label_list)))
predicted_emotion_sets = emotions_df[emotions_df['poem_id'].isin(gold_emotion_sets)].groupby('poem_id')['emotion_category'].apply(lambda values: {normalize_value(value) for value in values.dropna() if normalize_value(value)}).to_dict()
emotion_summary, emotion_details = set_metrics(gold_emotion_sets, predicted_emotion_sets)
display(emotion_summary)
emotion_details.head(10)


,true_positive,false_positive,false_negative,precision,recall,f1
0,221,34,0,0.866667,1.0,0.928571


,poem_id,true_positive,false_positive,false_negative,gold_count,predicted_count,matched_items,missed_items,extra_items
0,poem_11269,3,0,0,3,3,"hope, joy, mortality",,
1,poem_05452,7,2,0,7,9,"despair, envy, freedom, hope, love, melancholy, resilience",,"jealousy, shame"
2,poem_12480,6,0,0,6,6,"freedom, joy, melancholy, peace, tenderness, transcendence",,
3,poem_06414,6,2,0,6,8,"anxiety, freedom, gratitude, melancholy, mortality, peace",,"desire, longing"
4,poem_11839,2,1,0,2,3,"doubt, nostalgia",,anxiety
5,poem_09632,12,3,0,12,15,"despair, doubt, faith, fear, hope, love, mortality, nostalgia, regret, resilience, transcendence, wonder",,"grief, peace, tenderness"
6,poem_10031,8,0,0,8,8,"anxiety, fear, freedom, grief, hope, longing, love, nostalgia",,
7,poem_01828,7,0,0,7,7,"envy, fear, hope, melancholy, nostalgia, peace, transcendence",,
8,poem_01791,4,0,0,4,4,"desire, longing, peace, transcendence",,
9,poem_11907,2,0,0,2,2,"joy, transcendence",,


### Result Interpretation
Emotion extraction performs very strongly, with 221 true positives, 34 false positives, and no false negatives, giving a precision of 0.87, recall of 1.00, and F1 score of 0.93. This means the system successfully found every emotion category present in the gold sample, so its coverage is excellent. The lower precision compared with recall shows that it sometimes detects extra emotion categories that were not included in the gold labels, likely because the lexicon contains broad emotional words. Overall, the emotion extractor is the strongest part of the pipeline: it is highly reliable for capturing emotional language, though it could still be refined slightly to reduce over-detection.

## App Search Checks

The app search check is calculated live from graph files. It is not saved as a separate CSV file.


In [9]:
nodes_df = pd.read_csv(GRAPH_NODES_PATH)
relations_df = pd.read_csv(RELATIONS_PATH)
queries = ['moon', 'sea', 'grief', 'love', 'winter', 'light', 'death', 'memory', 'bird', 'loneliness']
app_rows = []
for query in queries:
    lowered = query.lower()
    node_matches = nodes_df[nodes_df['label'].fillna('').str.lower().str.contains(lowered, regex=False)]
    relation_matches = relations_df[
        relations_df['source_symbol'].fillna('').str.lower().str.contains(lowered, regex=False)
        | relations_df['target_emotion'].fillna('').str.lower().str.contains(lowered, regex=False)
    ]
    app_rows.append({
        'query': query,
        'matching_nodes': len(node_matches),
        'matching_relations': len(relation_matches),
        'top_related_node': node_matches['label'].iloc[0] if len(node_matches) else '',
        'example_poem_found': bool(len(relation_matches))
    })
pd.DataFrame(app_rows)


,query,matching_nodes,matching_relations,top_related_node,example_poem_found
0,moon,62,433,moon,True
1,sea,116,907,sea,True
2,grief,7,3004,grief,True
3,love,299,3340,glove,True
4,winter,68,23,winter,True
5,light,124,1755,light,True
6,death,78,850,death,True
7,memory,38,280,memory,True
8,bird,63,464,bird,True
9,loneliness,4,2600,loneliness,True


### Result Interpretation

The app search check is a practical sanity test for the deployed graph data. Each query should find at least some graph nodes or relations for common symbols and emotions that users are likely to search. This is not a precision/recall metric; it confirms that the processed graph files still support the main app workflows after filtering and regeneration.

## Summary Interpretation

The evaluation covers these parts of the project pipeline.

- Symbol extraction is evaluated with precision, recall, and F1 against the gold symbol sample.
- Emotion extraction is evaluated with precision, recall, and F1 against the gold emotion category sample.
- App search coverage is checked from the generated graph files to confirm that common user queries still return usable graph data.

The most important limitation is that these labels are objective proxy annotations, not expert literary annotations. For a stronger final report, the same annotation files can be manually reviewed by a human annotator and the notebook will still calculate the same metrics.